# HR Employee Attrition Analysis (Data Preparation)

**Business question:**
1. Berapa attrition rate keseluruhan dan bagaimana breakdown-nya per departemen, job role, dan job level?
2. Apakah overtime berkorelasi signifikan dengan attrition?
3. Apakah kompensasi (monthly income, salary hike, stock option) berbeda antara karyawan yang keluar vs bertahan?
4. Bagaimana pengaruh tenure & career stagnation terhadap attrition?
5. Bagaimana pengaruh kepuasan kerja terhadap attrition?
6. Profil risiko tinggi seperti apa yang terbentuk dari kombinasi faktor di atas?

**Dataset:** IBM HR Analytics Employee Attrition & Performance (Kaggle) — dataset yang dibuat oleh IBM data scientist, memiliki 1.470 baris, 35 kolom, berbentuk 1 tabel flat berisi data demografis, kompensasi, tenure, dan skor kepuasan kerja tiap karyawan.

**Tujuan notebook ini:** membersihkan data mentah dan memecahnya (normalisasi) dari 1 tabel menjadi 6 tabel relasional.

In [1]:
import pandas as pd

# Load dataset (sesuaikan nama filenya)
df = pd.read_csv('IBM_HR_Analytics_Employee_Attrition_&_Performance.csv', encoding='utf-8')
print(df.shape)
df.head()

(1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## 1. Data Quality Check

Sebelum normalisasi menjadi beberapa tabel, sebaiknya kita cek terlebih dahulu datanya, apakah ada missing value, duplikat, dan kolom yang nilainya konstan.

In [2]:
print(f'Missing values total: {df.isnull().sum().sum()}')
print(f'Duplikat total: {df.duplicated().sum()}')
print()
for column in df.columns:
    if df[column].nunique() == 1:
        print(f'Kolom {column} adalah nilai konstan')
    elif df[column].nunique() == 1470:
        print(f'Kolom {column} bisa dijadikan primary key')

Missing values total: 0
Duplikat total: 0

Kolom EmployeeCount adalah nilai konstan
Kolom EmployeeNumber bisa dijadikan primary key
Kolom Over18 adalah nilai konstan
Kolom StandardHours adalah nilai konstan


Tidak ada missing value dan tidak ada duplikat pada data ini, sehingga kolom `EmployeeNumber` bisa dijadikan primary key. Namun, terdapat 3 kolom (`EmployeeCount`, `Over18`, `StandardHours`) yang nilainya konstan di seluruh baris. Akibatnya, kolom tersebut bisa dihapus.

In [3]:
df = df.drop(['EmployeeCount', 'Over18', 'StandardHours'], axis=1)
df = df.rename(columns={'EmployeeNumber': 'employee_id'})
df.shape

(1470, 32)

## 2. Cek Relasi Job Role dengan Departemen

Apakah setiap `JobRole` selalu berada di satu `Department` yang sama, atau ada job role yang tersebar di beberapa departemen?

In [4]:
df.groupby('JobRole')['Department'].unique()

JobRole
Healthcare Representative                            [Research & Development]
Human Resources                                             [Human Resources]
Laboratory Technician                                [Research & Development]
Manager                      [Sales, Research & Development, Human Resources]
Manufacturing Director                               [Research & Development]
Research Director                                    [Research & Development]
Research Scientist                                   [Research & Development]
Sales Executive                                                       [Sales]
Sales Representative                                                  [Sales]
Name: Department, dtype: object

Job role **"Manager"** ternyata muncul di 3 separtemen berbeda. Ini berarti tabel `job_role` tidak bisa punya `department_id` sebagai foreign key tetap di tabel dimensinya. Akibatnya, `department_id` dan `job_role_id` sama-sama disimpan langsung di tabel `employees` sebagai foreign key terpisah.

## 3. Membangun Tabel Dimensi

Membuat 3 tabel dimensi dengan surrogate key(`_id` auto increment), untuk menggantikan nilai teks berulang di tabel utama (normalisasi dilakukan untuk mengurangi redudansi & mempermudah query di MySQL).

In [5]:
departments = pd.DataFrame({'department_name': sorted(df['Department'].unique())})
departments.index += 1
departments.index.name = 'department_id'
departments = departments.reset_index()

job_roles = pd.DataFrame({'job_role_name': sorted(df['JobRole'].unique())})
job_roles.index += 1
job_roles.index.name = 'job_role_id'
job_roles = job_roles.reset_index()

education_fields = pd.DataFrame({'field_name': sorted(df['EducationField'].unique())})
education_fields.index += 1
education_fields.index.name = 'education_field_id'
education_fields = education_fields.reset_index()

departments

,department_id,department_name
0,1,Human Resources
1,2,Research & Development
2,3,Sales


In [6]:
# Merge surrogate key ke tabel utama
df = df.merge(departments, left_on='Department', right_on='department_name', how='left')
df = df.merge(job_roles, left_on='JobRole', right_on='job_role_name', how='left')
df = df.merge(education_fields, left_on='EducationField', right_on='field_name', how='left')

assert df['department_id'].isnull().sum() == 0
assert df['job_role_id'].isnull().sum() == 0
assert df['education_field_id'].isnull().sum() == 0
print('Merge sukses, tidak ada nilai yang gagal ter-mapping')

Merge sukses, tidak ada nilai yang gagal ter-mapping


## 4. Membangun 3 Tabel Inti: `employees`, `compensation`, `satisfaction_scores`

- Tabel `employees` berisi data demografis & karier
- Tabel `compensation` berisi data gaji & kompensasi, berelasi satu satu dengan `employees`
- Tabel `satisfaction_scores` berisi data survei kepuasan & performance rating, berelasi satu satu dengan `employees`

In [7]:
employees = df[['employee_id','Age','Gender','MaritalStatus','Attrition','BusinessTravel',
                'OverTime','DistanceFromHome','Education','JobLevel','NumCompaniesWorked',
                'TotalWorkingYears','YearsAtCompany','YearsInCurrentRole',
                'YearsSinceLastPromotion','YearsWithCurrManager','TrainingTimesLastYear',
                'department_id','job_role_id','education_field_id']].copy()
employees.columns = ['employee_id','age','gender','marital_status','attrition','business_travel',
                     'over_time','distance_from_home','education_level','job_level',
                     'num_companies_worked','total_working_years','years_at_company',
                     'years_in_current_role','years_since_last_promotion',
                     'years_with_curr_manager','training_times_last_year',
                     'department_id','job_role_id','education_field_id']

compensation = df[['employee_id','DailyRate','HourlyRate','MonthlyIncome','MonthlyRate',
                    'PercentSalaryHike','StockOptionLevel']].copy()
compensation.columns = ['employee_id','daily_rate','hourly_rate','monthly_income',
                         'monthly_rate','percent_salary_hike','stock_option_level']

satisfaction_scores = df[['employee_id','EnvironmentSatisfaction','JobInvolvement',
                           'JobSatisfaction','RelationshipSatisfaction',
                           'WorkLifeBalance','PerformanceRating']].copy()
satisfaction_scores.columns = ['employee_id','environment_satisfaction','job_involvement',
                                'job_satisfaction','relationship_satisfaction',
                                'work_life_balance','performance_rating']

employees.head()


,employee_id,age,gender,marital_status,attrition,business_travel,over_time,distance_from_home,education_level,job_level,num_companies_worked,total_working_years,years_at_company,years_in_current_role,years_since_last_promotion,years_with_curr_manager,training_times_last_year,department_id,job_role_id,education_field_id
0,1,41,Female,Single,Yes,Travel_Rarely,Yes,1,2,2,8,8,6,4,0,5,0,3,8,2
1,2,49,Male,Married,No,Travel_Frequently,No,8,1,2,1,10,10,7,1,7,3,2,7,2
2,4,37,Male,Single,Yes,Travel_Rarely,Yes,2,2,1,6,7,0,0,0,0,3,2,3,5
3,5,33,Female,Married,No,Travel_Frequently,Yes,3,4,1,1,8,8,7,3,0,3,2,7,2
4,7,27,Male,Married,No,Travel_Rarely,No,2,1,1,9,6,2,2,2,2,3,2,3,4


## 5. Validasi Akhir & Export ke CSV

Cek kembali jumlah baris setiap tabel harus konsisten (yaitu 1.470 karyawan di semua tabel yang berelasi satu satu), lalu export ke CSV.

In [8]:
import os
os.makedirs('normalized_csv', exist_ok=True)

print('departments          :', departments.shape)
print('job_roles            :', job_roles.shape)
print('education_fields     :', education_fields.shape)
print('employees            :', employees.shape)
print('compensation         :', compensation.shape)
print('satisfaction_scores  :', satisfaction_scores.shape)

departments[['department_id','department_name']].to_csv('normalized_csv/departments.csv', index=False)
job_roles[['job_role_id','job_role_name']].to_csv('normalized_csv/job_roles.csv', index=False)
education_fields[['education_field_id','field_name']].to_csv('normalized_csv/education_fields.csv', index=False)
employees.to_csv('normalized_csv/employees.csv', index=False)
compensation.to_csv('normalized_csv/compensation.csv', index=False)
satisfaction_scores.to_csv('normalized_csv/satisfaction_scores.csv', index=False)

print('\nSemua tabel berhasil di-export ke folder normalized_csv/')

departments          : (3, 2)
job_roles            : (9, 2)
education_fields     : (6, 2)
employees            : (1470, 20)
compensation         : (1470, 7)
satisfaction_scores  : (1470, 7)

Semua tabel berhasil di-export ke folder normalized_csv/


## Ringkasan

| Tabel | Jumlah baris | Peran |
|---|---|---|
| `departments` | 3 | Dimensi |
| `job_roles` | 9 | Dimensi |
| `education_fields` | 6 | Dimensi |
| `employees` | 1470 | Inti |
| `compensation` | 1470 | Relasi 1:1 dengan employees |
| `satisfaction_scores` | 1470 | Relasi 1:1 dengan employees |

Selanjutnya, import tabel yang sudah di-export tadi ke MySQL